In [1]:
import os, sys
import glob
import numpy as np
import h5py
import joblib
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sys.path.append("../src/classifiers")

import cca
from cca import CCA_Classifier

from pathlib import Path


In [ ]:
BASE_DIR     = Path('../eeg-mind-reading-challenge')
DATA_DIR     = Path('../data')
STIM_DIR     = BASE_DIR / 'stimuli_train'  
TRAIN_DIR    = BASE_DIR / 'trn'
FEAT_DIR     = DATA_DIR / 'features'       
TEMP_FEAT    = DATA_DIR / 'temp_features' 
ARTIFACT_DIR = DATA_DIR / 'artifacts'
H5_DIR       = Path('../data')  

os.makedirs(ARTIFACT_DIR, exist_ok=True)

SFREQ = 500
DEC = 5
N_FRAMES = 8

def features_for(ids, layer_path):
    feat_path = FEAT_DIR / f'video_{layer_path}.npz'
    z = np.load(feat_path)
    rows = []
    for sid in ids:
        key = f'{int(sid):05d}'
        assert key in z.files, f'No features found for stimulus {sid}'
        rows.append(np.array(z[key], dtype=np.float32))   
    return np.stack(rows)

vids = sorted(list(STIM_DIR.glob('*.mp4')))
print(f"Found {len(vids)} total videos in {STIM_DIR}")
all_stim_ids = np.arange(1, len(vids) + 1)

print("Loading all unique video features for global PCA...")
X_unique_alex = features_for(all_stim_ids, 'features.3')
X_unique_dino = features_for(all_stim_ids, 'blocks.6')
X_unique_clip = features_for(all_stim_ids, 'visual.transformer.resblocks.7')

print("Preprocessing unique video features...")
sc = StandardScaler().fit(X_unique_alex)
X_trn_alex_z = sc.transform(X_unique_alex)
sc2 = StandardScaler().fit(X_unique_dino)
X_trn_dino_z = sc2.transform(X_unique_dino)
sc3 = StandardScaler().fit(X_unique_clip)
X_trn_clip_z = sc3.transform(X_unique_clip)

X_trn_ensemble = np.hstack([X_trn_alex_z, X_trn_dino_z, X_trn_clip_z])

pca = PCA(n_components=0.95, svd_solver='full', random_state=42).fit(X_trn_ensemble)
X_unique_p = pca.transform(X_trn_ensemble)

sc4 = StandardScaler().fit(X_unique_p)
X_unique_s = sc4.transform(X_unique_p)

# Build the required lookup dictionary
video_feature_map = {int(sid): X_unique_s[i] for i, sid in enumerate(all_stim_ids)}
print("Video feature map successfully created!")

joblib.dump({
    'sc': sc, 'pca': pca, 'sc2': sc2, 'sc3': sc3, 'sc4': sc4}, 
    ARTIFACT_DIR / 'cca_sub-global.joblib', compress=3)


Found 1000 total videos in ../eeg-mind-reading-challenge/stimuli_train
Loading all unique video features for global PCA...
Preprocessing unique video features...


In [ ]:
# --- 2. Loop Through Each Subject Individually ---
embedding_sources = ['schubki_cbramod','schubki_labram', 'schubki_luna', 'schubki_labram_special']
for sub in range(1, 7):  
    print(f"\n==============================")
    print(f"Training Models for Subject {sub:02d}")
    print(f"==============================")
    
    subject_models = {}
    score_matrices = []
    Y_train_ref = None

    candidate_matrix = np.array([video_feature_map[int(sid)] for sid in all_stim_ids])

    for source_name in embedding_sources:
        h5_path = H5_DIR / source_name / f'sub-{sub}.h5'
        if not os.path.exists(h5_path):
            print(f"-> Skipping source '{source_name}' for Subject {sub} (file not found at {h5_path})")
            continue
            
        print(f"-> Training CCA model on source: {source_name}")

        with h5py.File(h5_path, 'r') as f:
            if 'X' in f and 'Y' in f:
                X_train = np.array(f['X'], dtype=np.float32)
                Y_train = np.array(f['Y'], dtype=int)

        print(X_train.shape)

        Y_train_ref = Y_train
        Z_train = np.array([video_feature_map[int(vid)] for vid in Y_train])

        model = CCA_Classifier()
        model.fit(X_train, Z_train)

        score_matrix = model.get_score_matrix(X_train, candidate_matrix)
        score_matrices.append(score_matrix)
        
        subject_models[source_name] = model
        
        order_idx = np.argsort(-score_matrix, axis=1)
        ranked = all_stim_ids[order_idx].astype(np.int32)
        top1 = (ranked[:, 0] == Y_train).mean()
        top5 = (ranked[:, :5] == Y_train[:, None]).any(axis=1).mean()
        print(f"   [{source_name}] Train -> Top-1: {top1:.4f} | Top-5: {top5:.4f}")

    if not score_matrices:
        print(f"Skipping Subject {sub:02d} (no valid embedding sources found)")
        continue

    # --- SCORE-LEVEL ENSEMBLING ---
    ensemble_score_matrix = np.mean(score_matrices, axis=0)
    ensemble_order_idx = np.argsort(-ensemble_score_matrix, axis=1)
    ensemble_ranked = all_stim_ids[ensemble_order_idx].astype(np.int32)
    
    ensemble_top1 = (ensemble_ranked[:, 0] == Y_train_ref).mean()
    ensemble_top5 = (ensemble_ranked[:, :5] == Y_train_ref[:, None]).any(axis=1).mean()
    print(f"\n🌟 Ensembled Subject {sub:02d} Train -> Top-1: {ensemble_top1:.4f} | Top-5: {ensemble_top5:.4f}")

    # Save all models for this subject as a bundle artifact
    joblib.dump({'models': subject_models, 'sources': list(subject_models.keys())}, 
                ARTIFACT_DIR / f'cca_ensemble_sub-{sub:02d}.joblib', compress=3)
    print(f"Saved ensembled CCA models for Subject {sub:02d}!")


Training Models for Subject 01
-> Training CCA model on source: schubki_cbramod
(968, 200)


/Users/yetkin/Desktop/Hackathon/.venv/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:104: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)


   [schubki_cbramod] Train -> Top-1: 0.9535 | Top-5: 0.9948
-> Training CCA model on source: schubki_labram
(968, 200)


/Users/yetkin/Desktop/Hackathon/.venv/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:104: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)


   [schubki_labram] Train -> Top-1: 0.6818 | Top-5: 0.8657
-> Training CCA model on source: schubki_luna
(968, 256)
   [schubki_luna] Train -> Top-1: 0.7242 | Top-5: 0.8915
-> Training CCA model on source: schubki_labram_special
(968, 200)


...

   [schubki_cbramod] Train -> Top-1: 0.9683 | Top-5: 0.9947
-> Training CCA model on source: schubki_labram
(947, 200)
   [schubki_labram] Train -> Top-1: 0.6230 | Top-5: 0.8659
-> Training CCA model on source: schubki_luna
(947, 256)
   [schubki_luna] Train -> Top-1: 0.3970 | Top-5: 0.6357
-> Training CCA model on source: schubki_labram_special
(947, 200)
   [schubki_labram_special] Train -> Top-1: 0.6019 | Top-5: 0.8733

🌟 Ensembled Subject 04 Train -> Top-1: 1.0000 | Top-5: 1.0000
Saved ensembled CCA models for Subject 04!

Training Models for Subject 05
-> Training CCA model on source: schubki_cbramod
(952, 200)
   [schubki_cbramod] Train -> Top-1: 0.9748 | Top-5: 0.9947
-> Training CCA model on source: schubki_labram
(952, 200)


/Users/yetkin/Desktop/Hackathon/.venv/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:104: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)


   [schubki_labram] Train -> Top-1: 0.6376 | Top-5: 0.8382
-> Training CCA model on source: schubki_luna
(952, 256)
   [schubki_luna] Train -> Top-1: 0.6544 | Top-5: 0.8582
-> Training CCA model on source: schubki_labram_special
(952, 200)


/Users/yetkin/Desktop/Hackathon/.venv/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:104: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)


   [schubki_labram_special] Train -> Top-1: 0.6765 | Top-5: 0.8372

🌟 Ensembled Subject 05 Train -> Top-1: 1.0000 | Top-5: 1.0000
Saved ensembled CCA models for Subject 05!

Training Models for Subject 06
-> Training CCA model on source: schubki_cbramod
(850, 200)
   [schubki_cbramod] Train -> Top-1: 0.8847 | Top-5: 0.9988
-> Training CCA model on source: schubki_labram
(850, 200)
   [schubki_labram] Train -> Top-1: 0.2776 | Top-5: 0.6188
-> Training CCA model on source: schubki_luna
(850, 256)
   [schubki_luna] Train -> Top-1: 0.5200 | Top-5: 0.7729
-> Training CCA model on source: schubki_labram_special
(850, 200)
   [schubki_labram_special] Train -> Top-1: 0.2165 | Top-5: 0.5435

🌟 Ensembled Subject 06 Train -> Top-1: 1.0000 | Top-5: 1.0000
Saved ensembled CCA models for Subject 06!


In [ ]:
BASE_DIR         = Path('../eeg-mind-reading-challenge')
DATA_DIR         = Path('../data')
TEST_DIR         = BASE_DIR / 'tst'
STIM_TEST_DIR    = BASE_DIR / 'stimuli_test'
TEST_STIM        = BASE_DIR / 'tst_stim'
FRAME_TEST_DIR   = DATA_DIR / 'frames_test'
FEAT_TEST_DIR    = DATA_DIR / 'features_test'
TEMP_FEAT        = DATA_DIR / 'temp_features'
ARTIFACT_DIR     = DATA_DIR / 'artifacts'
PREDICTIONS_DIR  = Path('../predictions')
H5_DIR           = Path('../data')

os.makedirs(ARTIFACT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

def features_for(ids, layer_path):
    feat_path = FEAT_TEST_DIR / f'video_{layer_path}.npz'
    z = np.load(feat_path)
    rows = []
    for sid in ids:
        key = f'{int(sid):05d}'
        assert key in z.files, f'No features found for stimulus {sid}'
        rows.append(np.array(z[key], dtype=np.float32))   
    return np.stack(rows)

global_artifact_path = ARTIFACT_DIR / 'cca_sub-global.joblib'
B = joblib.load(global_artifact_path)
sc, sc2, sc3, sc4, pca = B['sc'], B['sc2'], B['sc3'], B['sc4'], B['pca']

vids = sorted(list(STIM_TEST_DIR.glob('*.mp4')))
print(f"Found {len(vids)} total videos in {STIM_TEST_DIR}")
all_stim_ids = np.arange(1001, 1001 + len(vids))

print("Loading all unique video features for global PCA...")
X_unique_alex = features_for(all_stim_ids, 'features.3')
X_unique_dino = features_for(all_stim_ids, 'blocks.6')
X_unique_clip = features_for(all_stim_ids, 'visual.transformer.resblocks.7')

print("Preprocessing unique video features...")
X_trn_alex_z = sc.transform(X_unique_alex)
X_trn_dino_z = sc2.transform(X_unique_dino)
X_trn_clip_z = sc3.transform(X_unique_clip)
X_trn_ensemble = np.hstack([X_trn_alex_z, X_trn_dino_z, X_trn_clip_z])
X_unique_p = pca.transform(X_trn_ensemble)
X_unique_s = sc4.transform(X_unique_p)

# Build the required lookup dictionary
video_feature_map = {int(sid): X_unique_s[i] for i, sid in enumerate(all_stim_ids)}
print("Video feature map successfully created!")

candidate_matrix = np.array([video_feature_map[int(sid)] for sid in all_stim_ids])
n_test_trials = len(all_stim_ids)
print("candidate_matrix:", candidate_matrix.shape)



Found 102 total videos in ../eeg-mind-reading-challenge/stimuli_test
Loading all unique video features for global PCA...
Preprocessing unique video features...
Video feature map successfully created!
candidate_matrix: (102, 825)


In [ ]:
# --- Loop Through Each Subject for Inference ---
for sub in range(1, 7):
    bundle_path = ARTIFACT_DIR / f'cca_ensemble_sub-{sub:02d}.joblib'        
    if not bundle_path.exists():        
        print(f"\nSkipping Subject {sub:02d} (model bundle not found)")
        continue

    print(f"\n==============================")
    print(f"Testing Ensembled CCA for Subject {sub:02d}")
    print(f"==============================")

    bundle = joblib.load(bundle_path)
    
    subject_models = bundle['models']
    sources_to_eval = bundle['sources']

    score_matrices = []

    for source_name in sources_to_eval:
        h5_test_path = H5_DIR / f'{source_name}_test/sub-{sub}.h5'
        if not os.path.exists(h5_test_path):
            print(f"-> Test H5 for source '{source_name}_test/sub-{sub}.h5' not found. Skipping this source.")
            continue

        print(f"-> Evaluating using test source: {source_name}")
        with h5py.File(h5_test_path, 'r') as f:
            X_test = np.array(f['X'], dtype=np.float32)
            
        model = subject_models[source_name]
        score_matrix = model.get_score_matrix(X_test, candidate_matrix)
        score_matrices.append(score_matrix)

    # --- SCORE-LEVEL ENSEMBLING ---
    final_score_matrix = np.mean(score_matrices, axis=0)
    
    order_idx = np.argsort(-final_score_matrix, axis=1)
    ranked = all_stim_ids[order_idx].astype(np.int32)

    # Save submission candidate list
    output_sub_path = PREDICTIONS_DIR / f'sub-{sub:02d}_ranked_candidate_list.npy'
    np.save(output_sub_path, ranked)
    print(f"Saved ensembled ranked candidate list for Subject {sub:02d} to {output_sub_path}")

    # --- Optional Evaluation if Ground Truth Exists (e.g., Subjects 1-4) ---
    gt_path = TEST_STIM / f'sub-{sub:02d}_tst_stim.npy'
    if os.path.exists(gt_path):
        gt = np.load(gt_path)
        order = np.argsort(gt)
        truth_ids = gt[order]
        
        t = np.arange(n_test_trials)
        ranks = (-final_score_matrix).argsort(1).argsort(1)[t, t] + 1
        
        top1 = (ranked[:, 0] == truth_ids).mean()
        top5 = (ranked[:, :5] == truth_ids[:, None]).any(axis=1).mean()

        print(f'Evaluated Top-1: {top1:.4f} (chance {1/n_test_trials:.4f})')
        print(f'Evaluated Top-5: {top5:.4f} (chance {5/n_test_trials:.4f})')
        print(f'Median Rank: {np.median(ranks):.1f}')
    else:
        print(f"Subject {sub:02d}: Blind candidate list generated successfully (no ground truth available).")


Testing Ensembled CCA for Subject 01
-> Evaluating using test source: schubki_cbramod
-> Evaluating using test source: schubki_labram
-> Evaluating using test source: schubki_luna
-> Evaluating using test source: schubki_labram_special
Saved ensembled ranked candidate list for Subject 01 to ../predictions/sub-01_ranked_candidate_list.npy
Evaluated Top-1: 0.0098 (chance 0.0098)
Evaluated Top-5: 0.0784 (chance 0.0490)
Median Rank: 50.0

Testing Ensembled CCA for Subject 02
-> Evaluating using test source: schubki_cbramod
-> Evaluating using test source: schubki_labram
-> Evaluating using test source: schubki_luna
-> Evaluating using test source: schubki_labram_special
Saved ensembled ranked candidate list for Subject 02 to ../predictions/sub-02_ranked_candidate_list.npy
Evaluated Top-1: 0.0098 (chance 0.0098)
Evaluated Top-5: 0.0686 (chance 0.0490)
Median Rank: 57.0

Testing Ensembled CCA for Subject 03
-> Evaluating using test source: schubki_cbramod
-> Evaluating using test source: sc